# SAM 3 — segment anything you can name

SAM 2 follows what you point at. **SAM 3 also understands what things are**: type `"person"` and it finds every person in the frame, gives each one an id, and tracks them all — including people who walk in later, and people who leave and come back.

You can still point. A click, a box or a mask picks out one object exactly as in SAM 2, and that path is unchanged here.

### What this notebook covers

1. **Describe it** — track every instance of a phrase through a clip
2. **Point at it** — click, box or mask to pick exactly one object
3. **Both at once** — use a box to hint the search, not to name an object
4. **Real detections** — boxes and confidence scores from the image model
5. **SAM 3.1** — the newer variant, which tracks objects jointly
6. **When objects show up** — controlling how cautious the tracker is
7. **Several descriptions at once**

### What you need

- A CUDA GPU with roughly 8 GB free
- `checkpoints/sam3.pt` (~3.5 GB) — **access-gated by Meta**: request it at [ai.meta.com/sam](https://ai.meta.com/sam), then `pixi run download-sam3`
- Optional, for section 5: `checkpoints/sam3.1_multiplex.pt` — `pixi run download-sam3-1`

SAM 3 is a large model: budget a few seconds per frame, more on an older card. The clip below is trimmed to 16 frames so each tracking cell stays around a minute — raise `N_FRAMES` if you want longer runs.

## Which prompt do I want?

This is the one decision worth making up front. Start from what you are trying to do:

| I want to… | use | what happens |
|---|---|---|
| Find **every** car / person / dog in the video | `start_concept_session("car")` | the model searches every frame and tracks each match under its own id |
| Track **this one thing** I selected | `start_session()` + `GeometryPrompt.click` / `.box` / `.mask` | only your object is tracked, for the whole clip; nothing else can appear |
| Find everything similar to **this example** | `start_concept_session(...)` + `GeometryPrompt.concept_box` | the box steers the search on that frame; matches keep being tracked afterwards |

The first two are the common cases. The third looks like the second but behaves like the first — that trap is section 3.

## Set-up

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

if os.path.isdir("notebooks"):     # started from the repo root rather than notebooks/
    sys.path.insert(0, "notebooks")
import nb_utils as nb  # shared plotting helpers — see notebooks/nb_utils.py

nb.use_repo_root()     # so "configs/..." and "checkpoints/..." resolve
nb.use_dark(False)     # light figures; see nb_utils.use_dark for PyCharm's inversion
device = nb.pick_device(require_cuda=True)

nb.require_checkpoints(("checkpoints/sam3.pt", "pixi run download-sam3"))

### The clip

A bedroom, two children moving around. Everything below runs on these frames.

In [ ]:
from pathlib import Path

N_FRAMES = 16  # the clip has 200; 16 keeps every tracking cell to about a minute

frames = [
    np.asarray(Image.open(p).convert("RGB"))
    for p in sorted(Path("notebooks/videos/bedroom").glob("*.jpg"))[:N_FRAMES]
]
H, W, _ = frames[0].shape
print(f"{len(frames)} frames of {W}x{H}")

plt.figure(figsize=(7, 4))
plt.title("frame 0")
plt.imshow(frames[0])
plt.axis("off")
plt.show()

## 1. Describe what you want

Load the video model once, then open a **session** per video. The description is fixed when the session opens: it is what the model looks for on every frame, and it does not change mid-clip.

Then feed frames. Each call returns `{obj_id: result}` — one entry per object found and tracked so far.

In [ ]:
from sam.build_sam import build_sam3_video_predictor
from sam.prompts import ConceptPrompt, GeometryPrompt

video_predictor = build_sam3_video_predictor(
    config_file="configs/sam3/sam3.yaml",
    ckpt_path="checkpoints/sam3.pt",
    device="cuda",
)
print(f"loaded: {sum(p.numel() for p in video_predictor.parameters()) / 1e6:.0f}M params")

In [ ]:
CONCEPT = "person"

session = video_predictor.start_concept_session(CONCEPT)
tracked = nb.collect(session, frames)

print("object ids on the last frame:", sorted(tracked[-1]))
nb.show_frames(frames, tracked, f"tracking every '{CONCEPT}' in the clip",
               idxs=[0, 5, 10, 15], label_prefix=CONCEPT)

Two children, two ids, and nobody had to point at either of them. The ids are stable: an object that leaves the frame and comes back keeps the id it had before.

The phrase is just a phrase — swap `CONCEPT` for anything and re-run. A phrase that matches nothing simply returns nothing, and an object needs a few frames before the tracker commits to it (section 6), so very short runs under-report. Section 4 sweeps several phrases cheaply, on a single image.

### Objects that arrive later

Nothing about a description says *frame 0*. On this second clip the camera pans, a dancer walks out of frame, and later walks back in — the model picks them up again by itself, with no new prompt and with the id they had before.

We pick up this clip at its frame 80, so the cell covers the exit and the return without paying for the 80 frames before them.

In [ ]:
START = 80  # the dancer leaves around frame 90 and comes back around frame 97

dance = [
    np.asarray(Image.open(f"notebooks/videos/dance/{i:05d}.jpg").convert("RGB"))
    for i in range(START, 101)
]

session = video_predictor.start_concept_session("person")
danced = nb.collect(session, dance)

print("objects tracked at source frame:",
      {START + i: len(danced[i]) for i in (0, 5, 10, 17, 20)})
nb.show_frames(dance, danced, "a dancer leaves, then returns — picked up again automatically",
               idxs=[0, 10, 17, 20], label_prefix="person")

## 2. Point at one thing instead

Open a session with **no** description and the search never runs: only what you prompt exists, and it is tracked to the end of the clip. This is the SAM 2 interface, unchanged — the same three prompts work on a `Sam2VideoPredictor`.

### Click

In [ ]:
CLICK = (385.0, 230.0)  # a child, in 960x540 pixels

session = video_predictor.start_session()          # no description
clicked = nb.collect(session, frames, prompts_frame0=[GeometryPrompt.click(1, CLICK)])

nb.show_frames(
    frames, clicked, "one click on frame 0 — only that child is tracked",
    idxs=[0, 5, 10, 15], label_prefix="click",
    extra=lambda ax, i: nb.show_points([CLICK], [1], ax) if i == 0 else None,
)

### Box

A box selects one object the same way — it is read as its two corners. Use this when you already have a box from a detector or an annotation tool.

In [ ]:
BOX = (310, 0, 515, 397)  # xyxy pixels, around the girl

session = video_predictor.start_session()
boxed = nb.collect(session, frames, prompts_frame0=[GeometryPrompt.box(1, BOX)])

print("object ids by frame:", {i: sorted(boxed[i]) for i in (0, 8, 15)})
nb.show_frames(
    frames, boxed, "one box on frame 0 — one object, whole clip",
    idxs=[0, 5, 10, 15], label_prefix="box",
    extra=lambda ax, i: nb.show_box(BOX, ax, obj_id=5, label="prompt box", style="--")
    if i == 0 else None,
)

### Mask

If the first frame is already segmented — by an earlier run, another model, or a human — hand that mask over as the starting point.

Two limits: a mask cannot be combined with `concept_box` (section 3), and the SAM 3.1 model in section 5 does not accept mask prompts at all. Use this model when you want to seed from a mask.

In [ ]:
seed_mask = clicked[0][1]  # reuse the click result from above as a starting mask

session = video_predictor.start_session()
from_mask = nb.collect(session, frames, prompts_frame0=[GeometryPrompt.mask(1, seed_mask)])

nb.show_frames(frames, from_mask, "seeded from an existing mask",
               idxs=[0, 5, 10, 15], label_prefix="mask")

## 3. A box that steers the search

There is a second thing a box can mean: *find me things like this one*. That is `GeometryPrompt.concept_box`, and it behaves very differently from the box in section 2 — worth understanding before it surprises you.

| | `GeometryPrompt.box` (section 2) | `GeometryPrompt.concept_box` |
|---|---|---|
| what the box means | "track **this** object" | "search is biased **toward here**, on this frame" |
| needs a description | no | yes |
| searching afterwards | never | every frame |
| how many objects | exactly one, under your id | every match, ids assigned by the model |
| a negative box (`label=0`) | not allowed | "everything matching **except** this one" |

Because the search keeps running, the box is a hint for one frame and not a claim on one object. If you have no phrase to give, pass `PLACEHOLDER` and the model uses a generic caption — which is exactly what makes the next result look odd.

Below, one box around the girl yields **two** objects (the boy included), and by frame 15 the generic caption has stopped matching either of them, so both are dropped. That is the mode working as designed — it is a search, not a selection. When you want to track what you selected, use section 2.

In [ ]:
session = video_predictor.start_concept_session(video_predictor.PLACEHOLDER)
hinted = nb.collect(session, frames, prompts_frame0=[GeometryPrompt.concept_box(BOX)])

print("generic caption used:", repr(session.state.concept.prompt.text))
print("object ids by frame: ", {i: sorted(hinted[i]) for i in (0, 1, 15)})

nb.show_frames(
    frames, hinted, "box as a search hint — every match tracked, then dropped",
    idxs=[0, 1, 15], label_prefix="hint",
    extra=lambda ax, i: nb.show_box(BOX, ax, obj_id=5, label="hint box", style="--")
    if i == 0 else None,
)

Give it a real phrase instead of the generic caption and it behaves sensibly: the box steers which *person* is found first, and the search keeps tracking people afterwards.

In [ ]:
session = video_predictor.start_concept_session("person")
hinted = nb.collect(session, frames, prompts_frame0=[GeometryPrompt.concept_box(BOX)])

print("object ids by frame:", {i: sorted(hinted[i]) for i in (0, 8, 15)})
nb.show_frames(frames, hinted, "box hint + the phrase 'person'",
               idxs=[0, 5, 10, 15], label_prefix="person")

Asking for a `concept_box` in a session with no description raises rather than silently picking a caption for you — the two behaviours are too different to guess between.

In [ ]:
session = video_predictor.start_session()   # no description
try:
    session.process(frames[0], prompts=[GeometryPrompt.concept_box(BOX)])
except ValueError as e:
    print("ValueError:", e)

## 4. Real boxes and confidence scores

The video model returns masks, so every box drawn above was computed *from* a mask. The image model predicts boxes directly, along with a score per object and a `presence` value — how strongly the phrase matches the image at all.

Use it when you want detections on a single image rather than tracking.

In [ ]:
from sam.build_sam import build_sam3

nb.free(globals(), "video_predictor", "session")  # SAM 3 is large; free it first

image_predictor = build_sam3(
    config_file="configs/sam3/sam3.yaml",
    ckpt_path="checkpoints/sam3.pt",
    device="cuda",
)

det = image_predictor.predict(frames[0], ConceptPrompt("person"), confidence_threshold=0.5)
print(f"{det.num_detections} detection(s) | presence = {det.presence:.3f}")
print("boxes (xyxy pixels):")
print(det.boxes.cpu().numpy().round(1))

fig, ax = plt.subplots(figsize=(9, 6))
ax.set_title("predicted boxes and scores for 'person'")
ax.imshow(frames[0])
ax.axis("off")
for i in range(det.num_detections):
    nb.show_mask((det.masks_logits[i] > 0).cpu(), ax, obj_id=i)
    nb.show_box(det.boxes[i], ax, obj_id=i, label=f"{det.scores[i]:.2f}")
plt.show()

### Trading recall for precision

`confidence_threshold` is the gate. Lower it to catch more, raise it to keep only what the model is sure about.

In [ ]:
for phrase in ["person", "bed", "lamp", "window", "picture frame"]:
    d = image_predictor.predict(frames[0], ConceptPrompt(phrase), confidence_threshold=0.5)
    print(f"{phrase:<14} -> {d.num_detections:2d} detection(s) | presence = {d.presence:.3f}")

print()
for thr in [0.3, 0.5, 0.7, 0.9]:
    d = image_predictor.predict(frames[0], ConceptPrompt("person"), confidence_threshold=thr)
    print(f"threshold {thr} -> {d.num_detections} detection(s)")

In [ ]:
concepts = ["person", "bed", "lamp"]
fig, axes = plt.subplots(1, len(concepts), figsize=(6 * len(concepts), 5))
for ax, phrase in zip(np.atleast_1d(axes), concepts):
    d = image_predictor.predict(frames[0], ConceptPrompt(phrase), confidence_threshold=0.5)
    ax.set_title(f"'{phrase}' — {d.num_detections} found")
    ax.imshow(frames[0])
    ax.axis("off")
    for i in range(d.num_detections):
        nb.show_mask((d.masks_logits[i] > 0).cpu(), ax, obj_id=i)
        nb.show_box(d.boxes[i], ax, obj_id=i)
plt.tight_layout()
plt.show()

### "Everything matching, except this one"

A `concept_box` carries a sign. `label=1` (the default) pulls the search toward the box; `label=0` pushes it away — useful when one particular instance is a distraction you want excluded.

In [ ]:
for label, name in [(None, "no box"), (1, "positive box"), (0, "negative box")]:
    geom = None if label is None else GeometryPrompt.concept_box((300, 150, 470, 420), label=label)
    det = image_predictor.predict(frames[0], ConceptPrompt("person"),
                                  confidence_threshold=0.5, geometry=geom)
    scores = [round(float(s), 3) for s in det.scores.tolist()]
    print(f"{name:15s} -> {det.num_detections} detection(s), "
          f"presence {det.presence:.4f}, scores {scores}")

How hard the sign bites depends on the model. On this one it moves the scores and the presence value. On the SAM 3.1 model in the next section, a negative box removes the enclosed detection outright.

## 5. SAM 3.1 — tracking objects jointly

SAM 3.1 packs up to 16 objects into a single pass rather than tracking them one at a time. It is a similar size to the model above, not a bigger one. **The code is identical** — a different builder, the same sessions and the same prompts.

Differences worth knowing: it does not accept mask prompts, and a box-only prompt is less reliable than on the model above (it wants generous margin around the object; a tight crop often finds nothing).

> This section needs `checkpoints/sam3.1_multiplex.pt` (~3.5 GB). Skip it if you have not downloaded that one — nothing later depends on it.

In [ ]:
from sam.build_sam import build_sam3_multiplex_video_predictor

nb.require_checkpoints(("checkpoints/sam3.1_multiplex.pt", "pixi run download-sam3-1"))
nb.free(globals(), "image_predictor")

mux_predictor = build_sam3_multiplex_video_predictor(
    config_file="configs/sam3/sam3.1.yaml",
    ckpt_path="checkpoints/sam3.1_multiplex.pt",
    device="cuda",
)
print(f"loaded: {sum(p.numel() for p in mux_predictor.parameters()) / 1e6:.0f}M params")

In [ ]:
session = mux_predictor.start_concept_session(CONCEPT)
mux_tracked = nb.collect(session, frames)

print("object ids on the last frame:", sorted(mux_tracked[-1]))
nb.show_frames(frames, mux_tracked, f"SAM 3.1: every '{CONCEPT}'",
               idxs=[0, 5, 10, 15], label_prefix=CONCEPT)

Clicking works here too, exactly as in section 2 — and so does everything else in this notebook apart from mask prompts.

In [ ]:
session = mux_predictor.start_session()
mux_clicked = nb.collect(session, frames, prompts_frame0=[GeometryPrompt.click(1, CLICK)])

nb.show_frames(
    frames, mux_clicked, "SAM 3.1: one click, one object",
    idxs=[0, 5, 10, 15], label_prefix="click",
    extra=lambda ax, i: nb.show_points([CLICK], [1], ax) if i == 0 else None,
)

## 6. When does an object show up?

A tracker has to choose between reacting fast and reacting reliably. Show an object the instant it is spotted and you also show every false alarm; wait for confirmation and you miss its first frames.

`emit` picks the trade-off. It changes only what you are shown — every object is tracked and keeps its memory regardless.

| setting | shows |
|---|---|
| `Emit.CONFIRMED` (default) | objects seen for 3 frames running — no flicker, but a short delay |
| `Emit.VISIBLE` | objects the tracker currently believes in |
| `Emit.ALIVE` | everything, including objects it has started to doubt |

Each result also carries `tracklet_state`, if you would rather apply your own rule.

In [ ]:
from sam.results import Emit

# free whichever big model is still loaded — section 5 may have been skipped
nb.free(globals(), "mux_predictor", "image_predictor", "session")
video_predictor = build_sam3_video_predictor(
    config_file="configs/sam3/sam3.yaml", ckpt_path="checkpoints/sam3.pt", device="cuda",
)

for mode in (Emit.CONFIRMED, Emit.VISIBLE, Emit.ALIVE):
    video_predictor.emit = mode
    s = video_predictor.start_concept_session("person")
    counts = [len(m) for m in nb.collect(s, frames[:8])]
    print(f"{mode.value:10s} objects shown per frame (0-7): {counts}")

video_predictor.emit = Emit.CONFIRMED  # back to the default

## 7. Several descriptions at once

One session tracks one description. For several, open one session each and merge the results yourself — they run independently, so `"person"` and `"bed"` can never be confused for one another or steal each other's objects.

Each session numbers its objects from its own counter, so offset the ids before merging — pick an offset larger than the number of objects any one session will produce.

In [ ]:
CONCEPTS = ["person", "bed"]

sessions = {c: video_predictor.start_concept_session(c) for c in CONCEPTS}

merged = []
for frame in frames[:8]:
    frame_objs = {}
    for offset, (concept, s) in enumerate(sessions.items()):
        with torch.inference_mode():
            for obj_id, masklet in s.process(frame).items():
                frame_objs[offset * 5 + obj_id] = nb.to_mask(masklet)  # offset per concept
    merged.append(frame_objs)

print("objects per concept:", {c: len(s.state.bank.known_obj_ids) for c, s in sessions.items()})
nb.show_frames(frames, merged, "two descriptions, two sessions, merged for display",
               idxs=[0, 7], label_prefix="obj")

## Summary

| I want to… | code |
|---|---|
| Track every match of a phrase | `p.start_concept_session("dog")` then `session.process(frame)` |
| Track one object I picked | `p.start_session()` + `GeometryPrompt.click(1, (x, y))` |
| …from a box | `GeometryPrompt.box(1, (x0, y0, x1, y1))` |
| …from a mask | `GeometryPrompt.mask(1, mask)` |
| Steer the search with an example | `start_concept_session(phrase)` + `GeometryPrompt.concept_box(xyxy)` |
| Exclude one instance | `GeometryPrompt.concept_box(xyxy, label=0)` |
| Add an object mid-clip | prompt on any frame, not only the first — or let the search find it |
| Detections with real boxes | `build_sam3(...)` then `predict(image, ConceptPrompt(...))` |
| Track several phrases | one session per phrase, ids offset when merging |
| React sooner / more cautiously | `predictor.emit = Emit.VISIBLE` / `CONFIRMED` / `ALIVE` |

Masks come back as logits — `result.masks_logits > 0` is the binary mask, which is all `nb.to_mask` does.

### Limits worth knowing

- **One description per session.** It is fixed when the session opens and cannot be changed.
- **The generic caption is a poor substitute for a phrase.** A `concept_box` with `PLACEHOLDER` searches for something vague; give a real phrase when you have one.
- **Mask prompts are for this model only** — SAM 3.1 rejects them, and no model accepts a mask paired with a `concept_box`.
- **A description carries no negatives.** `ConceptPrompt` holds a phrase and nothing else; to exclude something, use a negative `concept_box`.
- **Long videos need memory care.** Per-object memory here is already windowed, so VRAM stays flat with clip length.

### Relationship to upstream SAM 3

This fork keeps Meta's weights and behaviour bit-for-bit; what it changes is the interface and the memory profile. Three differences are worth calling out if you have used the original:

- **Streaming.** Upstream loads the whole video before tracking; here you pass one frame per call and per-object memory is windowed, so VRAM does not grow with clip length.
- **Box routing is explicit.** Upstream silently adopts a generic caption the moment a box arrives without text, turning "track what I boxed" into "search for this caption forever". Here those are two different calls — section 2 versus section 3 — and asking for the second without a description raises.
- **Visibility is causal.** Upstream decides what to show by looking 15 frames ahead, which a streaming model cannot do. `Emit.CONFIRMED` is the causal half of that rule; `Emit.VISIBLE` matches what upstream's saved outputs show.

Behavioural equivalence with upstream is pinned by the tests under [`tests/parity/`](../tests/parity/).